We import the necessary libraries

In [1]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

The following line chooses between CPU and GPU if they are available.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

We will load the wine dataset data using Pandas. We will use LabelEncoder since the Wine category does not start at zero; it could be done differently, but this way is more general. Additionally, we will define features and labels and split them into train and test sets.

In [3]:
# Carga tus datos
df = pd.read_csv('Wine.csv')

# Tenemos que codificar porque las etiquetas no comienzan en 0
label_encoder = LabelEncoder()
df['Wine'] = label_encoder.fit_transform(df['Wine'])

# Separa las características (X) de las etiquetas (y)
X = df.drop('Wine', axis=1).values # Todas las columnas excepto la que queremos predecir
y = df['Wine'].values  # Solo la columna de la variable que queremos predecir

# Divide los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)


In [4]:
# Muestra el mapeo de clases a índices (opcional)
class_to_index = {label: index for index, label in enumerate(label_encoder.classes_)}
print("Mapeo de clase a índice:", class_to_index)

Mapeo de clase a índice: {1: 0, 2: 1, 3: 2}


We will normalize the training and test sets with StandardScaler. It's important to apply the same transformation to the test set as to the train set.

In [5]:
# Ahora normalizamos los conjuntos de entrenamiento y prueba
scaler = StandardScaler() 

# Ajusta el scaler solo a los datos de entrenamiento y transforma
X_train_scaled = scaler.fit_transform(X_train)

# Aplica la transformación al conjunto de prueba
X_test_scaled = scaler.transform(X_test)

We will now use two very important and useful tools in PyTorch: Dataset and Dataloader. Dataset is a class from which we will create a new class adapted to our data. In this new class, we need to define how the data is transformed into PyTorch tensors, the length of the dataset, and how to call an element of the dataset.

In [6]:
#TODO: Crea una clase Dataset para cargar tus datos
class Wine(Dataset):
    def __init__(self,features, labels): 
        self.features=torch.tensor(features,dtype=torch.float32)
        self.labels=torch.tensor(labels, dtype=torch.int64)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [7]:
# Ahora puedes crear tus instancias de Wine Dataset
train_dataset = Wine(X_train_scaled, y_train)
test_dataset =  Wine(X_test_scaled, y_test)


With the Dataloader, we will be able to define batches, that is, sets of data that are passed to the model to improve the parameters. An optimization step is performed for each batch. When all batches have been read, the entire dataset has been read, and this is called an "epoch".

In [8]:
# Crear el DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=32,shuffle=True) #batch_sample hyper parametro
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=True)

In [9]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

Here we will define a fully dense neural network model similar to the one studied previously. The difference is that now it is a multiclass problem, so we need to use a softmax type activation function followed by a cross-entropy loss. PyTorch applies softmax before the cross-entropy loss function, so it's not necessary to include it as the final activation function (that would be a mistake).

In [10]:
# Definir el modelo
class WineNet(nn.Module):
    def __init__(self):
        super(WineNet, self).__init__()
        self.fc1= nn.Linear(13,64) # 13 características de entrada
        self.fc2=nn.Linear(64,32)
        self.fc3=nn.Linear(32,3)# 3 clases de vino

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # No aplicamos softmax aquí
        return x

In [11]:
model = WineNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

Unlike in the previous LAB, we now have both a training and a test set, as well as batches for each of them. Therefore, it's useful to make a train function and a test function that iterate over the batches, and then a separate loop for the epochs.

In [12]:
def train(dataloader, model, loss_fn, optimizer, device):
    model.train()  # Pone el modelo en modo de entrenamiento
    total_loss, correct, total = 0, 0, 0
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        
        # Calcula la predicción y la pérdida
        pred = model(X)
        loss = loss_fn(pred, y)
        
        # Cuenta predicciones correctas
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
        total += y.size(0)

        # Retropropaga el error y actualiza los pesos
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Train Loss: {total_loss / len(dataloader)}")
    print(f"Train Accuracy: {100 * correct / total}%")


In [13]:
def test(dataloader, model, loss_fn, device):
    model.eval()  # Pone el modelo en modo de evaluación
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():  # No calcula gradientes para las operaciones dentro de este bloque
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            
            # Cuenta predicciones correctas
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            total += y.size(0)

    print(f"Test Loss: {total_loss / len(dataloader)}")
    print(f"Test Accuracy: {100 * correct / total}%")


In this loop, we will read through the dataset completely as many times as the number of epochs we have indicated. Moreover, after each epoch, the model's accuracy on both the trainset and the testset is printed, so we can observe if it predicts correctly or if overfitting occurs.

In [14]:
num_epochs = 100

In [15]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test(test_dataloader, model, loss_fn, device)
    
print("Done!")

Epoch 1
-------------------------------
Train Loss: 1.0710797905921936
Train Accuracy: 53.225806451612904%
Test Loss: 1.0656965800694056
Test Accuracy: 61.111111111111114%
Epoch 2
-------------------------------
Train Loss: 1.0680509805679321
Train Accuracy: 54.03225806451613%
Test Loss: 1.0624822548457555
Test Accuracy: 62.96296296296296%
Epoch 3
-------------------------------
Train Loss: 1.0627001821994781
Train Accuracy: 58.87096774193548%
Test Loss: 1.0561340706689017
Test Accuracy: 66.66666666666667%
Epoch 4
-------------------------------
Train Loss: 1.0561951100826263
Train Accuracy: 63.70967741935484%
Test Loss: 1.0477974925722395
Test Accuracy: 68.51851851851852%
Epoch 5
-------------------------------
Train Loss: 1.0486615300178528
Train Accuracy: 67.74193548387096%
Test Loss: 1.0386119740349906
Test Accuracy: 74.07407407407408%
Epoch 6
-------------------------------
Train Loss: 1.0408606231212616
Train Accuracy: 70.16129032258064%
Test Loss: 1.0323042188371931
Test Accurac